# UAVDT Notebook 05 — Dynamic BEV / 3D Scene Visualization

This notebook continues from Notebook 04. It loads metric tracks/cuboids, creates BEV and 3D animations, creates an interactive Plotly scene, and exports a dynamic scene package JSON.


In [1]:
#@title 1. Set local project paths

from pathlib import Path
import json
import math
import numpy as np
import pandas as pd
from notebook_local import resolve_project_dir, print_local_setup

PROJECT_DIR = resolve_project_dir()
NB04_DIR = PROJECT_DIR / 'work' / 'notebook_04_metric_scene_export'
NB05_DIR = PROJECT_DIR / 'work' / 'notebook_05_dynamic_scene_visualization'
NB05_DIR.mkdir(parents=True, exist_ok=True)

print_local_setup(PROJECT_DIR)
print('NB04_DIR:', NB04_DIR)
print('NB05_DIR:', NB05_DIR)


PROJECT_DIR: /Users/vash/Dev/ResearchLab/Work/Drone3D/local_uav_bev_project
DATASET_DIR: not found
Set DRONE3D_PROJECT_DIR and UAVDT_DATASET_DIR to override these defaults.
NB04_DIR: /Users/vash/Dev/ResearchLab/Work/Drone3D/local_uav_bev_project/work/notebook_04_metric_scene_export
NB05_DIR: /Users/vash/Dev/ResearchLab/Work/Drone3D/local_uav_bev_project/work/notebook_05_dynamic_scene_visualization


In [2]:
#@title 2. Import visualization tools

import matplotlib.pyplot as plt
import matplotlib.patches as patches
from IPython.display import HTML, display
import imageio.v2 as imageio
import plotly.graph_objects as go

def read_rgb_frame(path, target_shape=None):
    """Read a PNG and return a fixed-size RGB array for GIF writing."""
    img = imageio.imread(path)
    if img.ndim == 2:
        img = np.repeat(img[..., None], 3, axis=2)
    elif img.shape[2] == 4:
        alpha = img[..., 3:4].astype(np.float32) / 255.0
        rgb = img[..., :3].astype(np.float32)
        img = (rgb * alpha + 255.0 * (1.0 - alpha)).astype(np.uint8)
    else:
        img = img[..., :3]

    if target_shape is None or img.shape == target_shape:
        return img

    out = np.full(target_shape, 255, dtype=np.uint8)
    h = min(target_shape[0], img.shape[0])
    w = min(target_shape[1], img.shape[1])
    out[:h, :w, :] = img[:h, :w, :]
    return out

def read_gif_frames(frame_paths):
    raw = [read_rgb_frame(p) for p in frame_paths]
    if not raw:
        return []
    max_h = max(im.shape[0] for im in raw)
    max_w = max(im.shape[1] for im in raw)
    target_shape = (max_h, max_w, 3)
    return [read_rgb_frame(p, target_shape) for p in frame_paths]

print('Visualization imports OK')


Visualization imports OK


In [3]:
#@title 3. Load metric tracks and cuboids from Notebook 04

def find_first_existing(candidates, glob_dir=None, glob_patterns=None):
    for p in candidates:
        p = Path(p)
        if p.exists():
            return p
    if glob_dir is not None and glob_patterns is not None:
        for pattern in glob_patterns:
            found = sorted(Path(glob_dir).glob(pattern))
            if found:
                return found[0]
    return None

metric_tracks_path = find_first_existing(
    [NB04_DIR / 'vehicle_tracks_metric.csv', NB04_DIR / 'tracks_metric.csv', NB04_DIR / 'metric_tracks.csv'],
    glob_dir=NB04_DIR,
    glob_patterns=['*track*metric*.csv', '*metric*track*.csv']
)

cuboids_path = find_first_existing(
    [NB04_DIR / 'vehicle_cuboids_metric.csv', NB04_DIR / 'cuboids_metric.csv', NB04_DIR / 'metric_cuboids.csv'],
    glob_dir=NB04_DIR,
    glob_patterns=['*cuboid*metric*.csv', '*metric*cuboid*.csv']
)

if metric_tracks_path is None:
    raise FileNotFoundError('Could not find metric tracks CSV from Notebook 04.')

print('Metric tracks path:', metric_tracks_path)
metric = pd.read_csv(metric_tracks_path)
print('Metric tracks shape:', metric.shape)
print('Metric columns:', metric.columns.tolist())
display(metric.head())

if cuboids_path is None:
    print('No cuboids CSV found. We will create cuboids from metric tracks.')
    cuboids = None
else:
    print('Cuboids path:', cuboids_path)
    cuboids = pd.read_csv(cuboids_path)
    print('Cuboids shape:', cuboids.shape)
    print('Cuboid columns:', cuboids.columns.tolist())
    display(cuboids.head())


Metric tracks path: /Users/vash/Dev/ResearchLab/Work/Drone3D/local_uav_bev_project/work/notebook_04_metric_scene_export/vehicle_tracks_metric.csv
Metric tracks shape: (632, 44)
Metric columns: ['frame_index', 'target_id', 'bbox_left', 'bbox_top', 'bbox_width', 'bbox_height', 'out_of_view', 'occlusion', 'object_category', 'x1', 'y1', 'x2', 'y2', 'class_name', 'track_id', 'sample_frame_idx', 'sample_name', 'original_frame_index', 'frame_id', 'frame_idx', 'frame', 'confidence', 'bbox_cx', 'bbox_ground_y', 'bev_x_px', 'bev_y_px', 'bev_box_width_px', 'bev_box_height_px', 'inside_bev_canvas', 'x_m', 'y_m', 'metric_source', 't_s', 'x_m_raw', 'x_straightening_slope', 'x_straightening_applied', 'vx_mps', 'vy_mps', 'speed_mps', 'yaw_rad', 'heading_mode', 'track_dx_m', 'track_dy_m', 'heading_quality']


,frame_index,target_id,bbox_left,bbox_top,bbox_width,bbox_height,out_of_view,occlusion,object_category,x1,...,x_straightening_slope,x_straightening_applied,vx_mps,vy_mps,speed_mps,yaw_rad,heading_mode,track_dx_m,track_dy_m,heading_quality
0,1,4,934,174,89,59,3,1,1,934.0,...,-0.170722,True,0.961699,6.507957,6.578630,1.570796,road_axis_by_motion,-0.37658,42.616714,road_axis_by_motion
1,6,4,907,164,108,59,3,1,1,907.0,...,-0.170722,True,0.968643,9.199850,9.250703,1.570796,road_axis_by_motion,-0.37658,42.616714,road_axis_by_motion
2,11,4,878,155,105,52,1,1,1,878.0,...,-0.170722,True,-0.115380,11.125092,11.125690,1.570796,road_axis_by_motion,-0.37658,42.616714,road_axis_by_motion
3,16,4,854,146,95,49,1,1,1,854.0,...,-0.170722,True,-1.299633,11.159353,11.234777,1.570796,road_axis_by_motion,-0.37658,42.616714,road_axis_by_motion
4,21,4,830,139,85,44,1,1,1,830.0,...,-0.170722,True,-1.542528,10.503418,10.616081,1.570796,road_axis_by_motion,-0.37658,42.616714,road_axis_by_motion


Cuboids path: /Users/vash/Dev/ResearchLab/Work/Drone3D/local_uav_bev_project/work/notebook_04_metric_scene_export/vehicle_cuboids_metric.csv
Cuboids shape: (632, 11)
Cuboid columns: ['frame_id', 'track_id', 'class_name', 'center_x_m', 'center_y_m', 'center_z_m', 'length_m', 'width_m', 'height_m', 'yaw_rad', 'speed_mps']


,frame_id,track_id,class_name,center_x_m,center_y_m,center_z_m,length_m,width_m,height_m,yaw_rad,speed_mps
0,0,4,car,26.597759,4.992191,0.75,4.5,1.8,1.5,1.570796,6.578630
1,1,4,car,26.717972,5.805686,0.75,4.5,1.8,1.5,1.570796,9.250703
2,2,4,car,26.839920,7.292154,0.75,4.5,1.8,1.5,1.570796,11.125690
3,3,4,car,26.689127,8.586959,0.75,4.5,1.8,1.5,1.570796,11.234777
4,4,4,car,26.515012,10.081992,0.75,4.5,1.8,1.5,1.570796,10.616081


In [4]:
#@title 4. Normalize columns and create cuboids if needed

def col(df, candidates, required=True):
    for c in candidates:
        if c in df.columns:
            return c
    if required:
        raise KeyError('Missing any of columns: ' + str(candidates) + '. Available: ' + str(df.columns.tolist()))
    return None

track_col = col(metric, ['track_id', 'id', 'object_id'])
frame_col = col(metric, ['frame_id', 'frame_idx', 'frame'])
x_col = col(metric, ['x_m', 'x_meter', 'x'])
y_col = col(metric, ['y_m', 'y_meter', 'y'])
yaw_col = col(metric, ['yaw_rad', 'yaw', 'heading_rad'], required=False)
class_col = col(metric, ['class_name', 'class', 'label'], required=False)

metric_norm = metric.copy()
metric_norm['track_id'] = metric_norm[track_col].astype(int)
metric_norm['frame_id'] = metric_norm[frame_col].astype(int)
metric_norm['x_m'] = pd.to_numeric(metric_norm[x_col], errors='coerce')
metric_norm['y_m'] = pd.to_numeric(metric_norm[y_col], errors='coerce')
metric_norm['yaw_rad'] = pd.to_numeric(metric_norm[yaw_col], errors='coerce') if yaw_col else 0.0
metric_norm['class_name'] = metric_norm[class_col].astype(str) if class_col else 'car'
metric_norm = metric_norm.dropna(subset=['x_m', 'y_m']).copy()

def default_dims_for_class(name):
    name = str(name).lower()
    if 'bus' in name:
        return 12.0, 2.6, 3.2
    if 'truck' in name:
        return 8.0, 2.5, 3.0
    return 4.5, 1.8, 1.5

if cuboids is None:
    rows = []
    for _, r in metric_norm.iterrows():
        length, width, height = default_dims_for_class(r['class_name'])
        rows.append({
            'frame_id': int(r['frame_id']),
            'track_id': int(r['track_id']),
            'class_name': r['class_name'],
            'center_x_m': float(r['x_m']),
            'center_y_m': float(r['y_m']),
            'center_z_m': float(height / 2.0),
            'length_m': float(length),
            'width_m': float(width),
            'height_m': float(height),
            'yaw_rad': float(r['yaw_rad']) if np.isfinite(r['yaw_rad']) else 0.0,
        })
    cuboids_norm = pd.DataFrame(rows)
else:
    cuboids_norm = cuboids.copy()
    if 'center_x_m' not in cuboids_norm.columns:
        cuboids_norm['center_x_m'] = pd.to_numeric(cuboids_norm[col(cuboids_norm, ['x_m', 'x'])], errors='coerce')
    if 'center_y_m' not in cuboids_norm.columns:
        cuboids_norm['center_y_m'] = pd.to_numeric(cuboids_norm[col(cuboids_norm, ['y_m', 'y'])], errors='coerce')
    if 'center_z_m' not in cuboids_norm.columns:
        cuboids_norm['center_z_m'] = pd.to_numeric(cuboids_norm.get('height_m', 1.5), errors='coerce') / 2.0
    if 'frame_id' not in cuboids_norm.columns:
        cuboids_norm['frame_id'] = cuboids_norm[col(cuboids_norm, ['frame_idx', 'frame'])].astype(int)
    if 'track_id' not in cuboids_norm.columns:
        cuboids_norm['track_id'] = cuboids_norm[col(cuboids_norm, ['id', 'object_id'])].astype(int)
    if 'class_name' not in cuboids_norm.columns:
        cuboids_norm['class_name'] = 'car'
    if 'yaw_rad' not in cuboids_norm.columns:
        cuboids_norm['yaw_rad'] = 0.0
    for name, fallback in [('length_m', 4.5), ('width_m', 1.8), ('height_m', 1.5)]:
        if name not in cuboids_norm.columns:
            cuboids_norm[name] = fallback

cuboids_norm = cuboids_norm.dropna(subset=['center_x_m', 'center_y_m']).copy()
cuboids_norm['frame_id'] = cuboids_norm['frame_id'].astype(int)
cuboids_norm['track_id'] = cuboids_norm['track_id'].astype(int)

metric_norm_path = NB05_DIR / 'metric_tracks_normalized.csv'
cuboids_norm_path = NB05_DIR / 'metric_cuboids_normalized.csv'
metric_norm.to_csv(metric_norm_path, index=False)
cuboids_norm.to_csv(cuboids_norm_path, index=False)

print('Saved normalized metric tracks:', metric_norm_path)
print('Saved normalized cuboids:', cuboids_norm_path)
print('Frames:', int(metric_norm['frame_id'].min()), 'to', int(metric_norm['frame_id'].max()))
print('Tracks:', metric_norm['track_id'].nunique())
display(cuboids_norm.head())


Saved normalized metric tracks: /Users/vash/Dev/ResearchLab/Work/Drone3D/local_uav_bev_project/work/notebook_05_dynamic_scene_visualization/metric_tracks_normalized.csv
Saved normalized cuboids: /Users/vash/Dev/ResearchLab/Work/Drone3D/local_uav_bev_project/work/notebook_05_dynamic_scene_visualization/metric_cuboids_normalized.csv
Frames: 0 to 39
Tracks: 23


,frame_id,track_id,class_name,center_x_m,center_y_m,center_z_m,length_m,width_m,height_m,yaw_rad,speed_mps
0,0,4,car,26.597759,4.992191,0.75,4.5,1.8,1.5,1.570796,6.578630
1,1,4,car,26.717972,5.805686,0.75,4.5,1.8,1.5,1.570796,9.250703
2,2,4,car,26.839920,7.292154,0.75,4.5,1.8,1.5,1.570796,11.125690
3,3,4,car,26.689127,8.586959,0.75,4.5,1.8,1.5,1.570796,11.234777
4,4,4,car,26.515012,10.081992,0.75,4.5,1.8,1.5,1.570796,10.616081


In [5]:
#@title 5. Helper functions for oriented cuboids

def rectangle_corners_2d(cx, cy, length, width, yaw):
    dx = length / 2.0
    dy = width / 2.0
    local = np.array([[-dx, -dy], [dx, -dy], [dx, dy], [-dx, dy]], dtype=float)
    c = math.cos(yaw)
    s = math.sin(yaw)
    R = np.array([[c, -s], [s, c]], dtype=float)
    return local @ R.T + np.array([cx, cy], dtype=float)

def cuboid_vertices(cx, cy, length, width, height, yaw):
    bottom = rectangle_corners_2d(cx, cy, length, width, yaw)
    vertices = []
    for x, y in bottom:
        vertices.append([x, y, 0.0])
    for x, y in bottom:
        vertices.append([x, y, height])
    return np.asarray(vertices, dtype=float)

def cuboid_edges():
    return [(0, 1), (1, 2), (2, 3), (3, 0), (4, 5), (5, 6), (6, 7), (7, 4), (0, 4), (1, 5), (2, 6), (3, 7)]

def get_scene_limits(df):
    xs = df['center_x_m'].to_numpy(dtype=float)
    ys = df['center_y_m'].to_numpy(dtype=float)
    if len(xs) == 0:
        return (0, 50), (0, 80)
    pad_x = max(5.0, 0.1 * (xs.max() - xs.min() + 1e-6))
    pad_y = max(5.0, 0.1 * (ys.max() - ys.min() + 1e-6))
    return (max(0, xs.min() - pad_x), xs.max() + pad_x), (max(0, ys.min() - pad_y), ys.max() + pad_y)

xlim, ylim = get_scene_limits(cuboids_norm)
print('Scene xlim:', xlim)
print('Scene ylim:', ylim)


Scene xlim: (0, np.float64(35.28595718940982))
Scene ylim: (0, np.float64(97.99816899762878))


In [6]:
#@title 6. Create metric BEV animation

MAX_ANIMATION_FRAMES = 80 #@param {type:'integer'}
FRAME_STEP = 1 #@param {type:'integer'}

frames_all = sorted(cuboids_norm['frame_id'].unique().tolist())
frames_use = frames_all[::max(1, FRAME_STEP)][:MAX_ANIMATION_FRAMES]

bev_frame_dir = NB05_DIR / 'bev_animation_frames'
bev_frame_dir.mkdir(parents=True, exist_ok=True)

for frame_id in frames_use:
    fdf = cuboids_norm[cuboids_norm['frame_id'] == frame_id]
    fig, ax = plt.subplots(figsize=(8, 10))
    ax.set_title(f'Metric BEV vehicles, frame {frame_id}')
    ax.set_xlim(*xlim)
    ax.set_ylim(*ylim)
    ax.set_aspect('equal', adjustable='box')
    ax.grid(True, alpha=0.3)
    ax.set_xlabel('X meters')
    ax.set_ylabel('Y meters')

    for _, r in fdf.iterrows():
        cx = float(r['center_x_m'])
        cy = float(r['center_y_m'])
        length = float(r['length_m'])
        width = float(r['width_m'])
        yaw = float(r['yaw_rad']) if np.isfinite(r['yaw_rad']) else 0.0
        corners = rectangle_corners_2d(cx, cy, length, width, yaw)
        poly = patches.Polygon(corners, closed=True, fill=True, alpha=0.35)
        ax.add_patch(poly)
        ax.plot(cx, cy, marker='o', markersize=3)
        ax.text(cx, cy, str(int(r['track_id'])), fontsize=8)

    frame_path = bev_frame_dir / f'bev_frame_{frame_id:06d}.png'
    fig.tight_layout()
    fig.savefig(frame_path, dpi=140)
    plt.close(fig)

bev_gif_path = NB05_DIR / 'metric_bev_animation.gif'
bev_frame_paths = [bev_frame_dir / f'bev_frame_{fid:06d}.png' for fid in frames_use]
images = read_gif_frames(bev_frame_paths)
if images:
    imageio.mimsave(bev_gif_path, images, duration=0.2)

print('Saved BEV animation frames:', bev_frame_dir)
print('Saved BEV GIF:', bev_gif_path)
display(HTML(f'<img src="{bev_gif_path}" width="600">'))


Saved BEV animation frames: /Users/vash/Dev/ResearchLab/Work/Drone3D/local_uav_bev_project/work/notebook_05_dynamic_scene_visualization/bev_animation_frames
Saved BEV GIF: /Users/vash/Dev/ResearchLab/Work/Drone3D/local_uav_bev_project/work/notebook_05_dynamic_scene_visualization/metric_bev_animation.gif


In [7]:
#@title 7. Create 3D cuboid animation GIF

from mpl_toolkits.mplot3d.art3d import Poly3DCollection

zmax = max(4.0, float(cuboids_norm['height_m'].max()) * 3.0)
frame3d_dir = NB05_DIR / 'scene3d_animation_frames'
frame3d_dir.mkdir(parents=True, exist_ok=True)

faces_idx = [[0, 1, 2, 3], [4, 5, 6, 7], [0, 1, 5, 4], [1, 2, 6, 5], [2, 3, 7, 6], [3, 0, 4, 7]]

for frame_id in frames_use:
    fdf = cuboids_norm[cuboids_norm['frame_id'] == frame_id]
    fig = plt.figure(figsize=(9, 9))
    ax = fig.add_subplot(111, projection='3d')
    ax.set_title(f'Metric 3D cuboids, frame {frame_id}')

    road_x = [xlim[0], xlim[1], xlim[1], xlim[0], xlim[0]]
    road_y = [ylim[0], ylim[0], ylim[1], ylim[1], ylim[0]]
    road_z = [0, 0, 0, 0, 0]
    ax.plot(road_x, road_y, road_z)

    for _, r in fdf.iterrows():
        verts = cuboid_vertices(float(r['center_x_m']), float(r['center_y_m']), float(r['length_m']), float(r['width_m']), float(r['height_m']), float(r['yaw_rad']) if np.isfinite(r['yaw_rad']) else 0.0)
        faces = [[verts[i] for i in face] for face in faces_idx]
        poly = Poly3DCollection(faces, alpha=0.35, linewidths=0.8, edgecolors='k')
        ax.add_collection3d(poly)
        ax.text(float(r['center_x_m']), float(r['center_y_m']), float(r['height_m']) + 0.3, str(int(r['track_id'])), fontsize=8)

    ax.set_xlim(*xlim)
    ax.set_ylim(*ylim)
    ax.set_zlim(0, zmax)
    ax.set_xlabel('X meters')
    ax.set_ylabel('Y meters')
    ax.set_zlabel('Z meters')
    ax.set_box_aspect((xlim[1] - xlim[0], ylim[1] - ylim[0], zmax * 4.0))
    ax.view_init(elev=35, azim=-60)

    frame_path = frame3d_dir / f'scene3d_frame_{frame_id:06d}.png'
    fig.tight_layout()
    fig.savefig(frame_path, dpi=140)
    plt.close(fig)

scene3d_gif_path = NB05_DIR / 'metric_3d_cuboids_animation.gif'
scene3d_frame_paths = [frame3d_dir / f'scene3d_frame_{fid:06d}.png' for fid in frames_use]
images = read_gif_frames(scene3d_frame_paths)
if images:
    imageio.mimsave(scene3d_gif_path, images, duration=0.2)

print('Saved 3D animation frames:', frame3d_dir)
print('Saved 3D GIF:', scene3d_gif_path)
display(HTML(f'<img src="{scene3d_gif_path}" width="650">'))


Saved 3D animation frames: /Users/vash/Dev/ResearchLab/Work/Drone3D/local_uav_bev_project/work/notebook_05_dynamic_scene_visualization/scene3d_animation_frames
Saved 3D GIF: /Users/vash/Dev/ResearchLab/Work/Drone3D/local_uav_bev_project/work/notebook_05_dynamic_scene_visualization/metric_3d_cuboids_animation.gif


In [8]:
#@title 8. Interactive Plotly 3D scene for one frame

PLOTLY_FRAME_ID = int(frames_use[0]) if frames_use else 0 #@param {type:'integer'}

fdf = cuboids_norm[cuboids_norm['frame_id'] == PLOTLY_FRAME_ID]
fig = go.Figure()

fig.add_trace(go.Scatter3d(
    x=[xlim[0], xlim[1], xlim[1], xlim[0], xlim[0]],
    y=[ylim[0], ylim[0], ylim[1], ylim[1], ylim[0]],
    z=[0, 0, 0, 0, 0],
    mode='lines',
    name='road plane'
))

for _, r in fdf.iterrows():
    verts = cuboid_vertices(float(r['center_x_m']), float(r['center_y_m']), float(r['length_m']), float(r['width_m']), float(r['height_m']), float(r['yaw_rad']) if np.isfinite(r['yaw_rad']) else 0.0)
    edge_x = []
    edge_y = []
    edge_z = []
    for a, b in cuboid_edges():
        edge_x += [verts[a, 0], verts[b, 0], None]
        edge_y += [verts[a, 1], verts[b, 1], None]
        edge_z += [verts[a, 2], verts[b, 2], None]

    fig.add_trace(go.Scatter3d(
        x=edge_x,
        y=edge_y,
        z=edge_z,
        mode='lines',
        name=f"track {int(r['track_id'])}"
    ))

fig.update_layout(
    title=f'Interactive 3D cuboids, frame {PLOTLY_FRAME_ID}',
    scene=dict(xaxis_title='X meters', yaxis_title='Y meters', zaxis_title='Z meters', aspectmode='data'),
    width=900,
    height=750
)
fig.show()

plotly_html_path = NB05_DIR / f'interactive_scene_frame_{PLOTLY_FRAME_ID:06d}.html'
fig.write_html(plotly_html_path)
print('Saved interactive HTML:', plotly_html_path)


Saved interactive HTML: /Users/vash/Dev/ResearchLab/Work/Drone3D/local_uav_bev_project/work/notebook_05_dynamic_scene_visualization/interactive_scene_frame_000000.html


In [9]:
#@title 9. Export dynamic scene package JSON

if 'metric_norm_path' not in globals():
    metric_norm_path = NB05_DIR / 'metric_tracks_normalized.csv'
if 'cuboids_norm_path' not in globals():
    cuboids_norm_path = NB05_DIR / 'metric_cuboids_normalized.csv'
if 'bev_gif_path' not in globals():
    bev_gif_path = NB05_DIR / 'metric_bev_animation.gif'
if 'scene3d_gif_path' not in globals():
    scene3d_gif_path = NB05_DIR / 'metric_3d_cuboids_animation.gif'

scene_package = {
    'description': 'UAVDT dynamic metric scene from BEV projection and cuboids',
    'coordinate_system': {
        'x': 'BEV horizontal meters',
        'y': 'BEV vertical meters',
        'z': 'height meters',
        'road_plane_z': 0.0,
    },
    'scene_limits': {
        'x_min_m': float(xlim[0]),
        'x_max_m': float(xlim[1]),
        'y_min_m': float(ylim[0]),
        'y_max_m': float(ylim[1]),
    },
    'files': {
        'metric_tracks_normalized_csv': str(metric_norm_path),
        'metric_cuboids_normalized_csv': str(cuboids_norm_path),
        'bev_animation_gif': str(bev_gif_path),
        'scene3d_animation_gif': str(scene3d_gif_path),
    },
    'frames': [],
}

for frame_id in sorted(cuboids_norm['frame_id'].unique().tolist()):
    fdf = cuboids_norm[cuboids_norm['frame_id'] == frame_id]
    objects = []
    for _, r in fdf.iterrows():
        objects.append({
            'track_id': int(r['track_id']),
            'class_name': str(r['class_name']),
            'center_m': [float(r['center_x_m']), float(r['center_y_m']), float(r.get('center_z_m', r['height_m'] / 2.0))],
            'size_m': [float(r['length_m']), float(r['width_m']), float(r['height_m'])],
            'yaw_rad': float(r['yaw_rad']) if np.isfinite(r['yaw_rad']) else 0.0,
        })
    scene_package['frames'].append({'frame_id': int(frame_id), 'objects': objects})

scene_json_path = NB05_DIR / 'dynamic_scene_package.json'
with open(scene_json_path, 'w') as f:
    json.dump(scene_package, f, indent=2)

print('Saved dynamic scene package:', scene_json_path)
print('Number of frames:', len(scene_package['frames']))
print('Total objects:', sum(len(fr['objects']) for fr in scene_package['frames']))


Saved dynamic scene package: /Users/vash/Dev/ResearchLab/Work/Drone3D/local_uav_bev_project/work/notebook_05_dynamic_scene_visualization/dynamic_scene_package.json
Number of frames: 40
Total objects: 632


## Next notebook

Notebook 06 can improve accuracy by using UAVDT ground-truth tracks or a stronger tracker, and by improving homography/scale calibration.
